In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrow
import xlrd
import folium
from branca.colormap import LinearColormap
from branca.colormap import StepColormap
from shapely.ops import unary_union

In [ ]:
os.chdir("/Users/sunny/Library/CloudStorage/OneDrive-Personal/Documents/Python/Electoral maps project")

In [ ]:
pwd

In [ ]:
# Load in the India parliamentary constituencies shapefile
# NB a shapefile is really a group of files
# But you only need to specify the path to the .shp file, and gpd will load in the associated files
# as long as they are in the same folder
districts = gpd.read_file("maps-master/parliamentary-constituencies/india_pc_2019.shp")

In [ ]:
districts = districts.drop(["ST_CODE", "PC_CODE"], axis=1)

In [ ]:
districts.rename(columns = {'ST_NAME' : "State", 
                            'PC_NAME' : "Constituency", 
                            'Res' : "Reserved status"}, inplace=True)
districts

In [ ]:
districts_border = districts[districts['State'] != 'JHARKHAND']

In [ ]:
districts_border = districts_border.sort_values(by = ["State", "Constituency"], ascending=True, ignore_index=True)

In [ ]:
# SC, ST, General verified
# For simplicity and merge-friendliness, remove these from the constituency name
# NB specify regex=True, otherwise the replacement will be interpreted literally!
districts_border["Constituency"] = districts_border["Constituency"].str.replace(r"\(.*", "", regex=True).str.strip()

In [ ]:
# Plot the map, check that it makes sense
fig, ax = plt.subplots(figsize=(15, 15))
districts_border.boundary.plot(ax=ax, linewidth=0.15)
plt.show()

In [ ]:
# Check for invalid geometries
invalid_geometries = districts[~districts.is_valid]
print(f"Invalid geometries:\n{invalid_geometries}")


In [ ]:
#   State Constituency Reserved status  \
#123    GUJARAT        PATAN             GEN   
# 352  RAJASTHAN         PALI  

invalid = districts[districts['Constituency'].isin(['PATAN', 'PALI'])]

In [ ]:
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

invalid_outline = gpd.GeoDataFrame(
    geometry=[unary_union(invalid.geometry)],
    crs=invalid.crs
)

folium.GeoJson(
    invalid_outline.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
# Fix invalid geometries
districts['geometry'] = districts['geometry'].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)

***Jharkhand***

In [ ]:
# Filter geometries for Jharkhand
jharkhand = districts[districts['State'].isin(['JHARKHAND', 'BIHAR', 'WEST BENGAL'])]

In [ ]:
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

jharkhand_outline = gpd.GeoDataFrame(
    geometry=[unary_union(jharkhand.geometry)],
    crs=jharkhand.crs
)

folium.GeoJson(
    jharkhand_outline.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
# India as a whole
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

india_outline = gpd.GeoDataFrame(
    geometry=[unary_union(districts.geometry)],
    crs=districts.crs
)

folium.GeoJson(
    india_outline.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
# try fixing jharkhand 
# first project lat-lon to flat
districts_2 = districts.to_crs("EPSG:32643")  # Example UTM zone for India


In [ ]:
# address borders
districts_2['geometry'] = districts_2['geometry'].buffer(1).buffer(-1)

In [ ]:
# try simplification?
districts_3 = districts_2.copy()
districts_3['geometry'] = districts_3['geometry'].simplify(tolerance=10)

In [ ]:
districts_2 = districts_2.to_crs("EPSG:4326")
districts_3 = districts_3.to_crs("EPSG:4326")

In [ ]:
# Check india again, with buffer but w/o simplification
# India as a whole
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

india_outline_2 = gpd.GeoDataFrame(
    geometry=[unary_union(districts_2.geometry)],
    crs=districts_2.crs
)

folium.GeoJson(
    india_outline_2.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
# Check india again, with simplification
# India as a whole
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

india_outline = gpd.GeoDataFrame(
    geometry=[unary_union(districts_3.geometry)],
    crs=districts_3.crs
)

folium.GeoJson(
    india_outline.to_json(),
    style_function=lambda feature: {
        'color': 'red', 
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
# Filter geometries for Jharkhand
jharkhand = districts[districts['State'].isin(['JHARKHAND', 'BIHAR', 'WEST BENGAL'])]

# Plot Jharkhand boundaries
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

jharkhand_outline = gpd.GeoDataFrame(
    geometry=[unary_union(jharkhand.geometry)],
    crs=jharkhand.crs
)

folium.GeoJson(
    jharkhand_outline.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
india_boundary_no_jharkhand = districts_border.unary_union

In [ ]:
jharkhand_area = districts[districts['State'] == 'JHARKHAND'].unary_union

In [ ]:
india_boundary_final = india_boundary_no_jharkhand.difference(jharkhand_area)

In [ ]:
from shapely.geometry import mapping
india_boundary_gdf = gpd.GeoDataFrame(
    {'geometry': [india_boundary_final]}, 
    crs=districts.crs
)

In [ ]:
# India as a whole
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

folium.GeoJson(
    india_boundary_gdf.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
jharkhand_area.is_valid  # Should return True

In [ ]:
india_boundary = districts.unary_union

In [ ]:
jharkhand_area = districts[districts['State'] == 'JHARKHAND'].unary_union

In [ ]:
india_boundary_final = india_boundary.difference(jharkhand_area)

In [ ]:
projected_crs = "EPSG:32643"  # Example of a UTM projection for India
districts_proj = districts.to_crs(projected_crs)

districts_no_jharkhand_proj = districts_proj[districts_proj['State'] != 'JHARKHAND']
jharkhand_area_proj = districts_proj[districts_proj['State'] == 'JHARKHAND'].unary_union



In [ ]:
buffered_jharkhand_area = jharkhand_area_proj.buffer(0.0001)  # A very small buffer

In [ ]:
india_boundary_no_jharkhand_proj = districts_no_jharkhand_proj.unary_union
india_boundary_final_proj = india_boundary_no_jharkhand_proj.difference(buffered_jharkhand_area)

In [ ]:
india_boundary_final_proj.is_valid  # Should return True

In [ ]:
india_boundary_final = gpd.GeoDataFrame(
    {'geometry': [india_boundary_final_proj]}, 
    crs=projected_crs
).to_crs(districts.crs)

In [ ]:
# India as a whole
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles=None)

folium.GeoJson(
    india_boundary_final.to_json(),
    style_function=lambda feature: {
        'color': 'red',  # Highlight Jharkhand in red for debugging
        'weight': 1,
        'fillOpacity': 0,
    }
).add_to(m)

m

In [ ]:
india_outline = unary_union(districts['geometry'])

In [ ]:
# Extract only the exterior boundary
india_exterior = india_outline.exterior

In [ ]:
india_ex